# Pyr Nuclei Table Download

Download or load a stable local copy of the `c3_nuclei_v1` table from the `zheng_ca3` CAVE datastack at materialization version `195`.

If the local Parquet file already exists and `overwrite_existing` is `False`, the notebook loads the local file for inspection and does not query/download the table again.

## Parameters

In [1]:
import sys
from pathlib import Path


def find_project_root(start=Path.cwd().resolve()):
    for candidate in [start, *start.parents]:
        if (candidate / "notebooks" / "helpers").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise RuntimeError(
        "Could not find the Pyr project root. Expected a parent directory containing "
        "notebooks/helpers and data."
    )


project_root = find_project_root()
helpers_dir = project_root / "notebooks" / "helpers"
if str(helpers_dir) not in sys.path:
    sys.path.insert(0, str(helpers_dir))

from cave_auth import load_cave_token
from path_behavior import format_path, print_path

# Set to True to show full absolute paths in user-facing notebook output.
show_full_path = False

datastack_name = "zheng_ca3"
materialization_version = 195
table_name = "c3_nuclei_v1"

output_dir = project_root / "data" / "nuclei"
parquet_filename = f"{table_name}_mat{materialization_version}.parquet"
metadata_filename = f"{table_name}_mat{materialization_version}_metadata.json"

overwrite_existing = False


## Local setup and authentication

This notebook uses project-local helper functions for CAVE authentication and path handling. The CAVE token is loaded from the local environment without printing the token itself, and generated artifact paths are displayed relative to the project directory by default.

## CAVE Client

In [2]:
import json
from datetime import datetime, timezone

import pandas as pd
from caveclient import CAVEclient

cave_token, cave_token_source = load_cave_token(project_root)
print(f"CAVE token loaded: {bool(cave_token)}")

client = CAVEclient(datastack_name=datastack_name, auth_token=cave_token)
print(f"Connected datastack: {client.datastack_name}")


CAVE token loaded: True
Connected datastack: zheng_ca3


## Materialization Timestamp

In [3]:
materialization_timestamp = client.materialize.get_timestamp(materialization_version)
if hasattr(materialization_timestamp, "isoformat"):
    materialization_timestamp = materialization_timestamp.isoformat()

print(f"materialization version: {materialization_version}")
print(f"materialization timestamp: {materialization_timestamp}")


materialization version: 195
materialization timestamp: 2025-02-26T10:10:01.468822+00:00


## Local nuclei-table artifact

The nuclei table is stored locally as a Parquet file so that subsequent inspection can reuse the downloaded data without querying CAVE again. In the saved run shown here, an existing local Parquet artifact was found and reused rather than downloaded again.

The local artifact corresponds to the c3_nuclei_v1 table from the zheng_ca3 datastack at materialization version 195.

## Load Or Download Table

In [4]:
output_dir = Path(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

parquet_path = output_dir / parquet_filename
metadata_path = output_dir / metadata_filename

if parquet_path.exists() and not overwrite_existing:
    print("Existing nuclei table output found; keeping local file.")
    print_path("Parquet path", parquet_path, project_root, show_full_path)
    nuclei_df = pd.read_parquet(parquet_path)
    table_source = "local_existing_parquet"
    parquet_written = False
else:
    print(f"Querying {table_name} from {datastack_name} at materialization {materialization_version}...")
    nuclei_df = client.materialize.query_table(
        table_name,
        materialization_version=materialization_version,
    )
    nuclei_df.to_parquet(parquet_path, index=False)
    table_source = "cave_materialization_query"
    parquet_written = True
    print(f"Saved Parquet: {format_path(parquet_path, project_root, show_full_path)}")


Existing nuclei table output found; keeping local file.
Parquet path: data\nuclei\c3_nuclei_v1_mat195.parquet


## Lightweight Inspection

In [5]:
print(f"row count: {len(nuclei_df)}")
print(f"column count: {len(nuclei_df.columns)}")
print("column names:")
print(nuclei_df.columns.tolist())
print("dtypes:")
print(nuclei_df.dtypes.astype(str).to_dict())

display(nuclei_df.head())


row count: 35499
column count: 10
column names:
['id', 'created', 'superceded_id', 'valid', 'volume', 'pt_supervoxel_id', 'pt_root_id', 'pt_position', 'bb_start_position', 'bb_end_position']
dtypes:
{'id': 'Int64', 'created': 'datetime64[ns, UTC]', 'superceded_id': 'Int64', 'valid': 'boolean', 'volume': 'Float32', 'pt_supervoxel_id': 'Int64', 'pt_root_id': 'Int64', 'pt_position': 'object', 'bb_start_position': 'object', 'bb_end_position': 'object'}


,id,created,superceded_id,valid,volume,pt_supervoxel_id,pt_root_id,pt_position,bb_start_position,bb_end_position
0,20099,2025-02-20 18:43:53.115880+00:00,<NA>,True,0.104509,76567172461640457,648518346433938771,"[53360, 64608, 1103]","[None, None, None]","[None, None, None]"
1,23059,2025-02-20 18:43:55.053192+00:00,<NA>,True,2.642596,77131084286202290,648518346436811563,"[57632, 71840, 281]","[None, None, None]","[None, None, None]"
2,24798,2025-02-20 18:43:56.143105+00:00,<NA>,True,0.167962,77623184861837124,648518346438478107,"[61152, 68272, 1809]","[None, None, None]","[None, None, None]"
3,13183,2025-02-20 18:43:22.039071+00:00,<NA>,True,264.815735,75088054712378344,648518346450796819,"[42848, 54624, 1911]","[None, None, None]","[None, None, None]"
4,16387,2025-02-20 18:43:40.930762+00:00,<NA>,True,0.126904,75652104110407854,648518346451269875,"[47088, 62528, 1683]","[None, None, None]","[None, None, None]"


## Metadata sidecar

A JSON metadata file accompanies the local Parquet artifact and records information about the source table, datastack, materialization version and timestamp, table dimensions, and artifact paths. In the saved run shown here, an existing metadata file was preserved rather than overwritten.

These Parquet and JSON files are generated local data products and are not required to be included with the public notebook.

## Save Metadata

In [6]:
metadata = {
    "datastack": datastack_name,
    "table_name": table_name,
    "materialization_version": materialization_version,
    "materialization_timestamp": materialization_timestamp,
    "row_count": int(len(nuclei_df)),
    "column_count": int(len(nuclei_df.columns)),
    "column_names": nuclei_df.columns.tolist(),
    "dtypes": nuclei_df.dtypes.astype(str).to_dict(),
    "output_filename": parquet_filename,
    "output_path": str(parquet_path),
    "metadata_filename": metadata_filename,
    "metadata_path": str(metadata_path),
    "table_source": table_source,
    "parquet_written": parquet_written,
    "metadata_written_at": datetime.now(timezone.utc).isoformat(),
}

if metadata_path.exists() and not overwrite_existing:
    print(
        "Metadata JSON already exists and overwrite_existing is False; leaving unchanged: "
        f"{format_path(metadata_path, project_root, show_full_path)}"
    )
else:
    metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    print(f"Saved metadata JSON: {format_path(metadata_path, project_root, show_full_path)}")


Metadata JSON already exists and overwrite_existing is False; leaving unchanged: data\nuclei\c3_nuclei_v1_mat195_metadata.json


## Summary

In [7]:
print("Nuclei table local copy ready.")
print(f"Parquet path: {format_path(parquet_path, project_root, show_full_path)}")
print(f"metadata JSON path: {format_path(metadata_path, project_root, show_full_path)}")
print(f"DataFrame shape: {nuclei_df.shape}")


Nuclei table local copy ready.
Parquet path: data\nuclei\c3_nuclei_v1_mat195.parquet
metadata JSON path: data\nuclei\c3_nuclei_v1_mat195_metadata.json
DataFrame shape: (35499, 10)


## Lightweight nuclei table exploration

Descriptive checks on the locally loaded `c3_nuclei_v1` table before biological interpretation. These cells do not filter, classify, merge, or modify `nuclei_df`.

In [8]:
# Validity and missingness
if 'valid' in nuclei_df.columns:
    print('valid value counts:')
    display(nuclei_df['valid'].value_counts(dropna=False))
else:
    print("valid column not present")

print('null counts by column:')
display(nuclei_df.isna().sum().rename('null_count').to_frame())

if 'superceded_id' in nuclei_df.columns:
    non_null_superceded_count = int(nuclei_df['superceded_id'].notna().sum())
    print(f"non-null superceded_id values: {non_null_superceded_count}")
else:
    print("superceded_id column not present")

if 'volume' in nuclei_df.columns:
    missing_volume_count = int(nuclei_df['volume'].isna().sum())
    print(f"missing volume values: {missing_volume_count}")
else:
    print("volume column not present")

if 'pt_root_id' in nuclei_df.columns:
    missing_pt_root_id_count = int(nuclei_df['pt_root_id'].isna().sum())
    print(f"missing pt_root_id values: {missing_pt_root_id_count}")
else:
    print("pt_root_id column not present")


valid value counts:


valid
True    35499
Name: count, dtype: Int64

null counts by column:


,null_count
id,0
created,0
superceded_id,35499
valid,0
volume,0
pt_supervoxel_id,0
pt_root_id,0
pt_position,0
bb_start_position,0
bb_end_position,0


non-null superceded_id values: 0
missing volume values: 0
missing pt_root_id values: 0


In [9]:
# Identifier cardinality
identifier_columns = ['id', 'pt_supervoxel_id', 'pt_root_id']
identifier_summary = []

for column in identifier_columns:
    if column in nuclei_df.columns:
        identifier_summary.append({
            'column': column,
            'unique_non_null_values': int(nuclei_df[column].nunique(dropna=True)),
            'is_unique_including_nulls': bool(nuclei_df[column].is_unique),
            'is_unique_among_non_nulls': bool(nuclei_df[column].dropna().is_unique),
        })
    else:
        identifier_summary.append({
            'column': column,
            'unique_non_null_values': None,
            'is_unique_including_nulls': None,
            'is_unique_among_non_nulls': None,
        })

print(f"total rows: {len(nuclei_df)}")
display(pd.DataFrame(identifier_summary))


total rows: 35499


,column,unique_non_null_values,is_unique_including_nulls,is_unique_among_non_nulls
0,id,35499,True,True
1,pt_supervoxel_id,24124,False,False
2,pt_root_id,13874,False,False


In [10]:
# Rows per root ID
if 'pt_root_id' in nuclei_df.columns:
    root_counts = nuclei_df['pt_root_id'].value_counts(dropna=True)
    root_count_frequency = (
        root_counts.value_counts()
        .sort_index()
        .rename_axis('nuclei_rows_per_root')
        .reset_index(name='root_id_count')
    )
    top_root_counts = (
        root_counts.head(10)
        .rename_axis('pt_root_id')
        .reset_index(name='nuclei_rows')
    )

    print(f"root IDs represented: {len(root_counts)}")
    print(f"root IDs appearing exactly once: {int((root_counts == 1).sum())}")
    print(f"root IDs appearing more than once: {int((root_counts > 1).sum())}")
    print(f"maximum nuclei rows associated with one root ID: {int(root_counts.max()) if len(root_counts) else 0}")

    print('frequency distribution of nuclei rows per root:')
    display(root_count_frequency)

    print('top 10 root IDs by associated nuclei rows:')
    display(top_root_counts)
else:
    root_counts = None
    print("pt_root_id column not present; skipping rows-per-root analysis")


root IDs represented: 13874
root IDs appearing exactly once: 10379
root IDs appearing more than once: 3495
maximum nuclei rows associated with one root ID: 11245
frequency distribution of nuclei rows per root:


,nuclei_rows_per_root,root_id_count
0,1,10379
1,2,1500
2,3,662
3,4,404
4,5,285
5,6,210
6,7,140
7,8,97
8,9,61
9,10,37


top 10 root IDs by associated nuclei rows:


,pt_root_id,nuclei_rows
0,0,11245
1,648518346447755155,188
2,648518346442399751,153
3,648518346458497167,62
4,648518346454066613,54
5,648518346437928629,44
6,648518346447766769,37
7,648518346435381847,33
8,648518346444717749,32
9,648518346449540678,30


In [11]:
# Volume summary
if 'volume' in nuclei_df.columns:
    volume_series = pd.to_numeric(nuclei_df['volume'], errors='coerce').dropna()
    volume_quantiles = volume_series.quantile([0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
    volume_summary = pd.Series({
        'count': int(volume_series.count()),
        'min': volume_series.min(),
        '1st percentile': volume_quantiles.loc[0.01],
        '5th percentile': volume_quantiles.loc[0.05],
        '10th percentile': volume_quantiles.loc[0.10],
        '25th percentile': volume_quantiles.loc[0.25],
        'median': volume_quantiles.loc[0.50],
        '75th percentile': volume_quantiles.loc[0.75],
        '90th percentile': volume_quantiles.loc[0.90],
        '95th percentile': volume_quantiles.loc[0.95],
        '99th percentile': volume_quantiles.loc[0.99],
        'max': volume_series.max(),
        'mean': volume_series.mean(),
        'standard deviation': volume_series.std(),
    }).rename('volume')

    print('volume descriptive statistics:')
    display(volume_summary.to_frame())

    volume_columns = [column for column in ['id', 'pt_root_id', 'volume'] if column in nuclei_df.columns]
    print('10 smallest non-null volumes:')
    display(nuclei_df.loc[volume_series.index, volume_columns].sort_values('volume').head(10))

    print('10 largest non-null volumes:')
    display(nuclei_df.loc[volume_series.index, volume_columns].sort_values('volume', ascending=False).head(10))
else:
    volume_series = pd.Series(dtype='float64')
    print("volume column not present; skipping volume summary")


volume descriptive statistics:


,volume
count,35499.000000
min,0.003732
1st percentile,0.033592
5th percentile,0.044790
10th percentile,0.055987
25th percentile,0.111974
median,0.813681
75th percentile,7.394043
90th percentile,173.250522
95th percentile,642.921173


10 smallest non-null volumes:


,id,pt_root_id,volume
7134,16545,648518346457481961,0.003732
31211,21066,648518346448199154,0.011197
22603,20624,648518346442646925,0.011197
22215,35319,0,0.011197
29591,4471,648518346443706091,0.011197
22917,7608,648518346446223870,0.011197
13599,18942,648518346444054820,0.011197
24506,22661,648518346437788707,0.01493
21370,2521,0,0.01493
22946,7637,648518346437223174,0.01493


10 largest non-null volumes:


,id,pt_root_id,volume
23793,27858,0,2389.182861
29027,22876,648518346442419701,1677.171265
12014,17487,648518346436695831,1598.233032
23321,20043,648518346437232646,1597.512695
32216,21199,648518346442571721,1593.985474
5763,30064,0,1545.664795
2727,14593,648518346439575644,1542.197266
1640,11506,648518346440894353,1537.987061
29019,22868,648518346452485350,1533.257935
25145,9050,648518346442010597,1526.901611


## Coordinate and bounding-box inspection

The following checks distinguish between a coordinate column containing a non-null Python list object and that list actually containing valid coordinate values.

An initial structural check shows that bounding-box fields are present as list-like values. The later content inspection looks inside those lists and shows that the stored entries are `[None, None, None]`. The earlier result therefore indicates the presence of the list structure, not populated bounding-box coordinates.


In [12]:
# Bounding-box availability
def is_usable_coordinate_value(value):
    if value is None:
        return False
    try:
        is_missing = pd.isna(value)
        if isinstance(is_missing, bool):
            return not is_missing
    except Exception:
        pass
    try:
        return len(value) > 0
    except TypeError:
        return True

bbox_columns = ['bb_start_position', 'bb_end_position']
bbox_summary = []
for column in bbox_columns:
    if column in nuclei_df.columns:
        usable_count = int(nuclei_df[column].map(is_usable_coordinate_value).sum())
        bbox_summary.append({'column': column, 'usable_non_null_rows': usable_count})
    else:
        bbox_summary.append({'column': column, 'usable_non_null_rows': None})

display(pd.DataFrame(bbox_summary))


,column,usable_non_null_rows
0,bb_start_position,35499
1,bb_end_position,35499


In [13]:
# Position sanity check
if 'pt_position' in nuclei_df.columns:
    pt_position_null_count = int(nuclei_df['pt_position'].isna().sum())
    pt_position_non_null = nuclei_df['pt_position'].dropna()

    def coordinate_length(value):
        try:
            return len(value)
        except TypeError:
            return None

    pt_position_lengths = pt_position_non_null.map(coordinate_length)
    three_coordinate_count = int((pt_position_lengths == 3).sum())

    print(f"pt_position null values: {pt_position_null_count}")
    print('pt_position example values:')
    display(pt_position_non_null.head(5))
    print(f"non-null pt_position values with three coordinates: {three_coordinate_count} of {len(pt_position_non_null)}")
else:
    pt_position_null_count = None
    three_coordinate_count = None
    print("pt_position column not present; skipping position sanity check")


pt_position null values: 0
pt_position example values:


0    [53360, 64608, 1103]
1     [57632, 71840, 281]
2    [61152, 68272, 1809]
3    [42848, 54624, 1911]
4    [47088, 62528, 1683]
Name: pt_position, dtype: object

non-null pt_position values with three coordinates: 35499 of 35499


In [14]:
# Structural summary facts only
if 'pt_root_id' in nuclei_df.columns and root_counts is not None and len(root_counts):
    duplicated_root_fraction = float((root_counts > 1).mean())
    if duplicated_root_fraction > 0.5:
        root_duplication_summary = 'root IDs commonly have multiple nuclei rows'
    elif int((root_counts > 1).sum()) > 0:
        root_duplication_summary = 'most root IDs are one-to-one with nuclei rows, with some duplicates'
    else:
        root_duplication_summary = 'root IDs are one-to-one with nuclei rows among non-null pt_root_id values'
else:
    root_duplication_summary = 'root-ID duplication could not be assessed'

volume_populated_summary = (
    f"volume populated for {int(volume_series.count())} rows"
    if 'volume' in nuclei_df.columns
    else 'volume column not present'
)

if 'bb_start_position' in nuclei_df.columns and 'bb_end_position' in nuclei_df.columns:
    bb_start_count = int(nuclei_df['bb_start_position'].map(is_usable_coordinate_value).sum())
    bb_end_count = int(nuclei_df['bb_end_position'].map(is_usable_coordinate_value).sum())
    bbox_summary_text = f"bounding-box coordinates populated for start={bb_start_count}, end={bb_end_count} rows"
else:
    bbox_summary_text = 'one or both bounding-box columns not present'

if 'valid' in nuclei_df.columns:
    invalid_count = int((nuclei_df['valid'] == False).sum())
    invalid_summary = f"invalid rows present: {invalid_count}"
else:
    invalid_summary = 'valid column not present'

if 'superceded_id' in nuclei_df.columns:
    superseded_summary = f"non-null superceded_id rows: {int(nuclei_df['superceded_id'].notna().sum())}"
else:
    superseded_summary = 'superceded_id column not present'

print('Structural summary:')
print(f"- {root_duplication_summary}")
print(f"- {volume_populated_summary}")
print(f"- {bbox_summary_text}")
print(f"- {invalid_summary}")
print(f"- {superseded_summary}")


Structural summary:
- most root IDs are one-to-one with nuclei rows, with some duplicates
- volume populated for 35499 rows
- bounding-box coordinates populated for start=35499, end=35499 rows
- invalid rows present: 0
- non-null superceded_id rows: 0


## Follow-up structural diagnostics

Diagnostic-only checks for root ID `0`, duplicated nonzero root IDs, and the actual contents of bounding-box coordinate fields. These cells do not modify `nuclei_df`, write files, or merge external annotations.

In [ ]:
# Separate root ID 0 from nonzero roots
nuclei_root0 = nuclei_df[nuclei_df['pt_root_id'] == 0]
nuclei_nonzero = nuclei_df[nuclei_df['pt_root_id'] != 0]

nonzero_root_counts = nuclei_nonzero['pt_root_id'].value_counts(dropna=True)
duplicated_nonzero_root_counts = nonzero_root_counts[nonzero_root_counts > 1]
# Diagnostic: root IDs with the largest numbers of duplicated nonzero nuclei-table entries.
top_duplicated_nonzero_roots = duplicated_nonzero_root_counts.head(15)

print(f"rows with pt_root_id == 0: {len(nuclei_root0)}")
print(f"rows with nonzero pt_root_id: {len(nuclei_nonzero)}")
print(f"unique nonzero pt_root_id count: {nuclei_nonzero['pt_root_id'].nunique(dropna=True)}")
print(f"nonzero root IDs appearing once: {int((nonzero_root_counts == 1).sum())}")
print(f"nonzero root IDs appearing more than once: {int((nonzero_root_counts > 1).sum())}")
print(f"maximum rows associated with a nonzero root ID: {int(nonzero_root_counts.max()) if len(nonzero_root_counts) else 0}")

print('top 15 duplicated nonzero root IDs:')
display(
    top_duplicated_nonzero_roots
    .rename_axis('pt_root_id')
    .reset_index(name='nuclei_rows')
)


rows with pt_root_id == 0: 11245
rows with nonzero pt_root_id: 24254
unique nonzero pt_root_id count: 13873
nonzero root IDs appearing once: 10379
nonzero root IDs appearing more than once: 3494
maximum rows associated with a nonzero root ID: 188
top 15 duplicated nonzero root IDs:


,pt_root_id,nuclei_rows
0,648518346447755155,188
1,648518346442399751,153
2,648518346458497167,62
3,648518346454066613,54
4,648518346437928629,44
5,648518346447766769,37
6,648518346435381847,33
7,648518346444717749,32
8,648518346449540678,30
9,648518346448223041,29


In [16]:
print(top_duplicated_nonzero_roots.index.tolist())

[648518346447755155, 648518346442399751, 648518346458497167, 648518346454066613, 648518346437928629, 648518346447766769, 648518346435381847, 648518346444717749, 648518346449540678, 648518346448223041, 648518346441638397, 648518346449284164, 648518346431527510, 648518346430972250, 648518346444231783]


In [17]:
# Inspect duplicated nonzero roots
inspection_columns = [
    column for column in ['id', 'pt_root_id', 'pt_supervoxel_id', 'volume', 'pt_position']
    if column in nuclei_df.columns
]

duplicated_root_summaries = []
for root_id in top_duplicated_nonzero_roots.head(5).index.tolist():
    root_rows = nuclei_nonzero[nuclei_nonzero['pt_root_id'] == root_id]
    volume_values = pd.to_numeric(root_rows['volume'], errors='coerce') if 'volume' in root_rows.columns else pd.Series(dtype='float64')
    duplicated_root_summaries.append({
        'pt_root_id': int(root_id),
        'nucleus_rows': int(len(root_rows)),
        'unique_pt_supervoxel_id_count': int(root_rows['pt_supervoxel_id'].nunique(dropna=True)) if 'pt_supervoxel_id' in root_rows.columns else None,
        'min_volume': volume_values.min() if len(volume_values.dropna()) else None,
        'median_volume': volume_values.median() if len(volume_values.dropna()) else None,
        'max_volume': volume_values.max() if len(volume_values.dropna()) else None,
    })

    print(f"pt_root_id {int(root_id)} associated rows:")
    display(root_rows[inspection_columns].sort_values('volume', ascending=False) if 'volume' in inspection_columns else root_rows[inspection_columns])

print('top duplicated nonzero root summaries:')
display(pd.DataFrame(duplicated_root_summaries))


pt_root_id 648518346447755155 associated rows:


,id,pt_root_id,pt_supervoxel_id,volume,pt_position
31837,8997,648518346447755155,74455217051289757,126.971504,"[38192, 58000, 1994]"
11749,17234,648518346447755155,75931380063820781,79.4645,"[48944, 46224, 1669]"
2511,14377,648518346447755155,75438867373166006,79.378654,"[45456, 46800, 1078]"
27423,20837,648518346447755155,76987117385230450,79.218155,"[56432, 47840, 1836]"
24970,22583,648518346447755155,77269279489478772,75.392365,"[58800, 52864, 1504]"
...,...,...,...,...,...
23041,7732,648518346447755155,74103304275441888,0.033592,"[35792, 57216, 731]"
2340,14206,648518346447755155,75579123825106783,0.02986,"[46224, 42944, 1084]"
13428,18388,648518346447755155,76494330017479861,0.02986,"[53184, 46080, 1781]"
13436,18396,648518346447755155,76494261365189714,0.02986,"[52976, 45920, 2100]"


pt_root_id 648518346442399751 associated rows:


,id,pt_root_id,pt_supervoxel_id,volume,pt_position
13396,18878,648518346442399751,76285422472611362,90.452919,"[51488, 62752, 458]"
13338,18820,648518346442399751,76355310381495661,86.888405,"[52064, 58928, 1131]"
12948,18720,648518346442399751,76284529387900246,86.201622,"[51440, 55808, 1505]"
1666,11532,648518346442399751,75087848419839531,83.991997,"[42992, 52656, 1456]"
2853,14719,648518346442399751,75581322714063368,80.405083,"[46560, 59440, 497]"
...,...,...,...,...,...
2877,14743,648518346442399751,75581597524694553,0.026127,"[46464, 61440, 242]"
13284,18766,648518346442399751,76355241662079404,0.022395,"[51760, 58240, 1149]"
7013,16216,648518346442399751,75721441793902377,0.018662,"[47584, 54688, 624]"
5144,13237,648518346442399751,75370010725330662,0.018662,"[44816, 57920, 1890]"


pt_root_id 648518346458497167 associated rows:


...

,id,pt_root_id,pt_supervoxel_id,volume,pt_position
17377,28193,648518346454066613,78322886305286504,302.375671,"[66432, 38368, 726]"
16560,28143,648518346454066613,78322817854243009,204.383133,"[66320, 37856, 1733]"
25335,26653,648518346454066613,77970561682394569,81.927933,"[63696, 34752, 1261]"
17470,28286,648518346454066613,78463623927814938,70.125832,"[67232, 38288, 1174]"
27231,26812,648518346454066613,78041273889637638,69.808571,"[64176, 37728, 682]"
28496,25490,648518346454066613,77900124285645787,60.869286,"[63008, 34288, 1403]"
27239,26820,648518346454066613,78041205170412702,60.201168,"[64480, 36928, 830]"
27247,26828,648518346454066613,78111505261971640,43.472195,"[64560, 36464, 938]"
25324,26642,648518346454066613,78040999145987965,23.574343,"[64272, 35376, 1214]"
27363,26944,648518346454066613,78181942994092063,18.591482,"[65248, 36944, 1981]"


pt_root_id 648518346437928629 associated rows:


,id,pt_root_id,pt_supervoxel_id,volume,pt_position
26766,4248,648518346437928629,73189885326003852,79.490623,"[28896, 67664, 1924]"
23974,6727,648518346437928629,73822860157831991,76.750984,"[33344, 65024, 925]"
26532,6922,648518346437928629,73964490932659518,71.49192,"[34608, 71552, 832]"
28269,7963,648518346437928629,74174772463928310,70.390839,"[36176, 65696, 388]"
30727,9345,648518346437928629,74527440885790671,66.691956,"[38672, 71840, 706]"
23520,3587,648518346437928629,73119516515152560,2.459704,"[28592, 67584, 1871]"
26703,4185,648518346437928629,73260116564005967,1.690813,"[29616, 66800, 1637]"
27193,9234,648518346437928629,74386153574308987,1.254113,"[37776, 67744, 359]"
30679,9297,648518346437928629,74386359800294458,1.052559,"[37488, 69312, 830]"
19554,8047,648518346437928629,74104678799449686,1.052559,"[35824, 67952, 1344]"


top duplicated nonzero root summaries:


,pt_root_id,nucleus_rows,unique_pt_supervoxel_id_count,min_volume,median_volume,max_volume
0,648518346447755155,188,184,0.026127,0.158630,126.971504
1,648518346442399751,153,151,0.018662,0.160497,90.452919
2,648518346458497167,62,62,0.037325,0.477757,243.869049
3,648518346454066613,54,54,0.033592,2.685519,302.375671
4,648518346437928629,44,42,0.037325,0.218350,79.490623


In [18]:
# Correctly inspect bounding-box contents
def coordinate_missing_flags(value):
    try:
        values = list(value)
    except TypeError:
        values = [value]

    flags = []
    for item in values:
        try:
            flags.append(bool(pd.isna(item)))
        except Exception:
            flags.append(False)
    return flags


def coordinate_population_category(value):
    flags = coordinate_missing_flags(value)
    if len(flags) == 0:
        return 'all three coordinates missing'
    missing_count = sum(flags)
    if len(flags) == 3 and missing_count == 0:
        return 'all three coordinates populated'
    if len(flags) == 3 and missing_count == 3:
        return 'all three coordinates missing'
    return 'partially populated'

bbox_content_summaries = []
bbox_category_examples = {}
for column in ['bb_start_position', 'bb_end_position']:
    if column not in nuclei_df.columns:
        bbox_content_summaries.append({
            'column': column,
            'all three coordinates populated': None,
            'partially populated': None,
            'all three coordinates missing': None,
        })
        continue

    categories = nuclei_df[column].map(coordinate_population_category)
    counts = categories.value_counts().to_dict()
    bbox_content_summaries.append({
        'column': column,
        'all three coordinates populated': int(counts.get('all three coordinates populated', 0)),
        'partially populated': int(counts.get('partially populated', 0)),
        'all three coordinates missing': int(counts.get('all three coordinates missing', 0)),
    })

    examples = {}
    for category in [
        'all three coordinates populated',
        'partially populated',
        'all three coordinates missing',
    ]:
        example_rows = nuclei_df.loc[categories == category, ['id', 'pt_root_id', column]].head(5)
        if len(example_rows):
            examples[category] = example_rows
    bbox_category_examples[column] = examples

print('bounding-box content classification:')
display(pd.DataFrame(bbox_content_summaries))

for column, examples in bbox_category_examples.items():
    print(f"examples for {column}:")
    if not examples:
        print('  no examples available')
    for category, example_rows in examples.items():
        print(f"  {category}:")
        display(example_rows)


bounding-box content classification:


,column,all three coordinates populated,partially populated,all three coordinates missing
0,bb_start_position,0,0,35499
1,bb_end_position,0,0,35499


examples for bb_start_position:
  all three coordinates missing:


,id,pt_root_id,bb_start_position
0,20099,648518346433938771,"[None, None, None]"
1,23059,648518346436811563,"[None, None, None]"
2,24798,648518346438478107,"[None, None, None]"
3,13183,648518346450796819,"[None, None, None]"
4,16387,648518346451269875,"[None, None, None]"


examples for bb_end_position:
  all three coordinates missing:


,id,pt_root_id,bb_end_position
0,20099,648518346433938771,"[None, None, None]"
1,23059,648518346436811563,"[None, None, None]"
2,24798,648518346438478107,"[None, None, None]"
3,13183,648518346450796819,"[None, None, None]"
4,16387,648518346451269875,"[None, None, None]"


In [19]:
# Compare root-0 and nonzero volume distributions
def compact_volume_stats(df, label):
    volume = pd.to_numeric(df['volume'], errors='coerce').dropna()
    quantiles = volume.quantile([0.25, 0.50, 0.75, 0.90, 0.99]) if len(volume) else pd.Series(dtype='float64')
    return {
        'group': label,
        'count': int(volume.count()),
        '25th percentile': quantiles.get(0.25, None),
        'median': quantiles.get(0.50, None),
        '75th percentile': quantiles.get(0.75, None),
        '90th percentile': quantiles.get(0.90, None),
        '99th percentile': quantiles.get(0.99, None),
        'max': volume.max() if len(volume) else None,
    }

root0_nonzero_volume_summary = pd.DataFrame([
    compact_volume_stats(nuclei_root0, 'pt_root_id == 0'),
    compact_volume_stats(nuclei_nonzero, 'pt_root_id != 0'),
])

display(root0_nonzero_volume_summary)


,group,count,25th percentile,median,75th percentile,90th percentile,99th percentile,max
0,pt_root_id == 0,11245,0.093312,0.399375,2.892672,10.704753,373.985682,2389.182861
1,pt_root_id != 0,24254,0.123172,1.108547,53.460311,262.091754,1371.513860,1677.171265


In [20]:
# Follow-up structural summary facts only
duplicate_nonzero_count = int((nonzero_root_counts > 1).sum()) if len(nonzero_root_counts) else 0
duplicate_nonzero_fraction = float((nonzero_root_counts > 1).mean()) if len(nonzero_root_counts) else 0.0
if duplicate_nonzero_fraction > 0.10:
    duplicate_summary = f"duplicate-root behavior remains substantial after excluding root 0: {duplicate_nonzero_count} of {len(nonzero_root_counts)} nonzero roots ({duplicate_nonzero_fraction:.1%})"
elif duplicate_nonzero_count:
    duplicate_summary = f"duplicate-root behavior remains present after excluding root 0: {duplicate_nonzero_count} of {len(nonzero_root_counts)} nonzero roots ({duplicate_nonzero_fraction:.1%})"
else:
    duplicate_summary = "duplicate-root behavior was not observed after excluding root 0"

bbox_summary_df = pd.DataFrame(bbox_content_summaries).set_index('column')
bbox_populated_summary = (
    "bounding-box coordinates are actually populated"
    if (bbox_summary_df['all three coordinates populated'].fillna(0) > 0).any()
    else "bounding-box coordinate entries are not actually populated"
)

if duplicated_root_summaries:
    multiple_supervoxel_roots = sum(
        summary['unique_pt_supervoxel_id_count'] > 1
        for summary in duplicated_root_summaries
        if summary['unique_pt_supervoxel_id_count'] is not None
    )
    supervoxel_summary = f"among the top duplicated roots inspected, {multiple_supervoxel_roots} of {len(duplicated_root_summaries)} contain multiple distinct pt_supervoxel_id values"
else:
    supervoxel_summary = "no duplicated nonzero roots were available for pt_supervoxel_id inspection"

if len(root0_nonzero_volume_summary) == 2:
    root0_median = root0_nonzero_volume_summary.loc[root0_nonzero_volume_summary['group'] == 'pt_root_id == 0', 'median'].iloc[0]
    nonzero_median = root0_nonzero_volume_summary.loc[root0_nonzero_volume_summary['group'] == 'pt_root_id != 0', 'median'].iloc[0]
    root0_p99 = root0_nonzero_volume_summary.loc[root0_nonzero_volume_summary['group'] == 'pt_root_id == 0', '99th percentile'].iloc[0]
    nonzero_p99 = root0_nonzero_volume_summary.loc[root0_nonzero_volume_summary['group'] == 'pt_root_id != 0', '99th percentile'].iloc[0]
    volume_comparison_summary = f"root-0 vs nonzero volume summaries differ: medians {root0_median} vs {nonzero_median}; 99th percentiles {root0_p99} vs {nonzero_p99}"
else:
    volume_comparison_summary = "root-0 and nonzero volume distributions could not be compared"

print('Follow-up structural summary:')
print(f"- {duplicate_summary}")
print(f"- {bbox_populated_summary}")
print(f"- {supervoxel_summary}")
print(f"- {volume_comparison_summary}")


Follow-up structural summary:
- duplicate-root behavior remains substantial after excluding root 0: 3494 of 13873 nonzero roots (25.2%)
- bounding-box coordinate entries are not actually populated
- among the top duplicated roots inspected, 5 of 5 contain multiple distinct pt_supervoxel_id values
- root-0 vs nonzero volume summaries differ: medians 0.3993753492832184 vs 1.1085466146469116; 99th percentiles 373.98568237304335 vs 1371.5138598632816


### Comparison with local cell annotations

The remainder of the notebook is an exploratory comparison between the downloaded nuclei table and a companion CA3 cell-annotation CSV included with this repository. This analysis is not part of the nuclei-table download itself; it examines overlap between annotated cell roots and nuclei records, including roots associated with multiple nuclei rows and populations without matching annotations.

The annotation CSV provides the local reference dataset used for these comparisons.

In [21]:
# Load and inspect the local annotation CSV
annotation_csv_path = project_root / "data" / "mouse_hippocampus_ca3_cell_annotations_export.csv"
annotation_df = pd.read_csv(annotation_csv_path)

print(f"annotation CSV path: {format_path(annotation_csv_path, project_root, show_full_path)}")
print(f"annotation shape: {annotation_df.shape}")
print("annotation columns:")
print(annotation_df.columns.tolist())
print("annotation dtypes:")
print(annotation_df.dtypes.astype(str).to_dict())

display(annotation_df.head())


annotation CSV path: data\mouse_hippocampus_ca3_cell_annotations_export.csv
annotation shape: (2764, 10)
annotation columns:
['Cell ID', 'SV ID', 'x', 'y', 'z', 'type', 'subtypes', 'outputs', 'inputs', 'Links']
annotation dtypes:
{'Cell ID': 'int64', 'SV ID': 'int64', 'x': 'int64', 'y': 'int64', 'z': 'int64', 'type': 'object', 'subtypes': 'object', 'outputs': 'float64', 'inputs': 'float64', 'Links': 'object'}


,Cell ID,SV ID,x,y,z,type,subtypes,outputs,inputs,Links
0,648518346439010972,73539941870724508,31440,54240,144,pyramidal cell,NaN,NaN,NaN,https://spelunker.cave-explorer.org/#!%7B%22di...
1,648518346447880508,75226798866163813,43568,39312,150,pyramidal cell,NaN,NaN,NaN,https://spelunker.cave-explorer.org/#!%7B%22di...
2,648518346450503598,75930348868987954,49072,38768,152,pyramidal cell,NaN,NaN,NaN,https://spelunker.cave-explorer.org/#!%7B%22di...
3,648518346450505390,76282398748329608,51536,39952,156,pyramidal cell,NaN,NaN,NaN,https://spelunker.cave-explorer.org/#!%7B%22di...
4,648518346437067786,75578230270219832,46112,36544,162,pyramidal cell,sparsely thorny,NaN,NaN,https://spelunker.cave-explorer.org/#!%7B%22di...


In [22]:
# Identify the annotation column corresponding to CAVE/root ID
id_candidate_rows = []
priority_names = {
    'pt_root_id': 100,
    'root_id': 95,
    'root id': 95,
    'cell id': 90,
    'cell_id': 90,
    'cave id': 80,
    'cave_id': 80,
    'segment id': 75,
    'segment_id': 75,
    'sv id': 20,
    'sv_id': 20,
    'pt_supervoxel_id': 20,
}

for column in annotation_df.columns:
    normalized_name = str(column).strip().lower().replace('-', ' ').replace('_', ' ')
    compact_name = normalized_name.replace(' ', '_')
    name_score = priority_names.get(normalized_name, priority_names.get(compact_name, 0))
    if 'root' in normalized_name and 'id' in normalized_name:
        name_score = max(name_score, 85)
    if 'cell' in normalized_name and 'id' in normalized_name:
        name_score = max(name_score, 80)
    if 'sv' in normalized_name and 'id' in normalized_name:
        name_score = max(name_score, 15)

    numeric_values = pd.to_numeric(annotation_df[column], errors='coerce')
    non_null_count = int(numeric_values.notna().sum())
    large_id_count = int((numeric_values.dropna().abs() > 1_000_000_000_000).sum())
    unique_non_null_count = int(numeric_values.nunique(dropna=True))
    id_like_score = name_score + (10 if large_id_count else 0) + (5 if unique_non_null_count else 0)

    if name_score or large_id_count:
        id_candidate_rows.append({
            'column': column,
            'name_score': name_score,
            'non_null_numeric_values': non_null_count,
            'large_id_values': large_id_count,
            'unique_non_null_numeric_values': unique_non_null_count,
            'selection_score': id_like_score,
        })

id_candidate_df = pd.DataFrame(id_candidate_rows).sort_values(
    ['selection_score', 'large_id_values', 'unique_non_null_numeric_values'],
    ascending=False,
)

if id_candidate_df.empty:
    raise RuntimeError('Could not identify a CAVE/root-ID-like column in the annotation CSV.')

annotation_root_id_column = id_candidate_df.iloc[0]['column']
print('candidate annotation ID columns:')
display(id_candidate_df)
print(f"selected annotation root ID column: {annotation_root_id_column}")


candidate annotation ID columns:


,column,name_score,non_null_numeric_values,large_id_values,unique_non_null_numeric_values,selection_score
0,Cell ID,90,2764,2763,2764,105
1,SV ID,20,2764,2763,2764,35


selected annotation root ID column: Cell ID


In [ ]:
# Annotation coverage against nuclei_df['pt_root_id']
# Diagnostic: annotated root IDs associated with the largest numbers of nuclei rows.
annotation_root_id_series = pd.to_numeric(annotation_df[annotation_root_id_column], errors='coerce').dropna().astype('int64')
annotation_root_ids = set(int(value) for value in annotation_root_id_series.tolist())

nuclei_root_id_series = pd.to_numeric(nuclei_df['pt_root_id'], errors='coerce').dropna().astype('int64')
nuclei_nonzero_root_ids = set(int(value) for value in nuclei_root_id_series[nuclei_root_id_series != 0].tolist())

matched_annotation_root_ids = sorted(annotation_root_ids & nuclei_nonzero_root_ids)
unmatched_annotation_root_ids = sorted(annotation_root_ids - nuclei_nonzero_root_ids)
match_percentage = (len(matched_annotation_root_ids) / len(annotation_root_ids) * 100) if annotation_root_ids else 0.0

print(f"total annotation rows: {len(annotation_df)}")
print(f"unique annotation root IDs: {len(annotation_root_ids)}")
print(f"annotated roots occurring in nuclei_df: {len(matched_annotation_root_ids)}")
print(f"annotated roots not occurring in nuclei_df: {len(unmatched_annotation_root_ids)}")
print(f"percentage matched: {match_percentage:.2f}%")
print(f"unmatched_annotation_root_ids = {unmatched_annotation_root_ids}")


total annotation rows: 2764
unique annotation root IDs: 2764
annotated roots occurring in nuclei_df: 2061
annotated roots not occurring in nuclei_df: 703
percentage matched: 74.57%
unmatched_annotation_root_ids = [0, 648518346417172254, 648518346423530999, 648518346426190839, 648518346426198775, 648518346426244343, 648518346426266871, 648518346426529942, 648518346426813115, 648518346427243152, 648518346427259280, 648518346427529666, 648518346427684546, 648518346428338816, 648518346428768896, 648518346428974720, 648518346429330740, 648518346429426588, 648518346429451714, 648518346429582422, 648518346429657943, 648518346429661378, 648518346429877791, 648518346430028219, 648518346430111007, 648518346430334651, 648518346430541844, 648518346430577536, 648518346430780029, 648518346430976384, 648518346431329699, 648518346431401123, 648518346431458723, 648518346431485237, 648518346431548191, 648518346431559711, 648518346431562271, 648518346431647319, 648518346431647990, 648518346431653566, 648

In [24]:
# Nucleus rows per annotated root
nuclei_counts_by_root = nuclei_df[nuclei_df['pt_root_id'] != 0]['pt_root_id'].value_counts(dropna=True)
annotation_nucleus_counts = pd.Series(
    {root_id: int(nuclei_counts_by_root.get(root_id, 0)) for root_id in sorted(annotation_root_ids)},
    name='nucleus_rows',
)
annotation_nucleus_frequency = (
    annotation_nucleus_counts.value_counts()
    .sort_index()
    .rename_axis('nucleus_rows_per_annotated_root')
    .reset_index(name='annotated_root_count')
)
annotation_nucleus_bucket_counts = {
    '0': int((annotation_nucleus_counts == 0).sum()),
    '1': int((annotation_nucleus_counts == 1).sum()),
    '2': int((annotation_nucleus_counts == 2).sum()),
    '3+': int((annotation_nucleus_counts >= 3).sum()),
}
max_annotated_nucleus_rows = int(annotation_nucleus_counts.max()) if len(annotation_nucleus_counts) else 0

top_annotated_roots_by_nuclei = (
    annotation_nucleus_counts.sort_values(ascending=False).head(15)
    .rename_axis('pt_root_id')
    .reset_index(name='nucleus_rows')
)

print('nucleus rows per annotated root buckets:')
print(annotation_nucleus_bucket_counts)
print(f"maximum nucleus rows for an annotated root: {max_annotated_nucleus_rows}")
print('frequency distribution:')
display(annotation_nucleus_frequency)
print('top 15 annotated roots by nucleus rows:')
display(top_annotated_roots_by_nuclei)


nucleus rows per annotated root buckets:
{'0': 703, '1': 417, '2': 406, '3+': 1238}
maximum nucleus rows for an annotated root: 16
frequency distribution:


,nucleus_rows_per_annotated_root,annotated_root_count
0,0,703
1,1,417
2,2,406
3,3,338
4,4,249
5,5,197
6,6,150
7,7,108
8,8,80
9,9,46


top 15 annotated roots by nucleus rows:


,pt_root_id,nucleus_rows
0,648518346448199666,16
1,648518346462161795,16
2,648518346443172484,16
3,648518346454880649,15
4,648518346438316205,14
5,648518346429301812,14
6,648518346443147652,14
7,648518346446645962,14
8,648518346446991251,13
9,648518346440948518,13


In [25]:
print(top_annotated_roots_by_nuclei.pt_root_id.tolist())

[648518346448199666, 648518346462161795, 648518346443172484, 648518346454880649, 648518346438316205, 648518346429301812, 648518346443147652, 648518346446645962, 648518346446991251, 648518346440948518, 648518346448327390, 648518346442391506, 648518346442089733, 648518346438029017, 648518346448049658]


In [26]:
# Known versus unlabeled nuclei populations
nuclei_nonzero_for_annotation = nuclei_df[nuclei_df['pt_root_id'] != 0]
nuclei_annotated = nuclei_nonzero_for_annotation[nuclei_nonzero_for_annotation['pt_root_id'].isin(annotation_root_ids)]
nuclei_unlabeled_nonzero = nuclei_nonzero_for_annotation[~nuclei_nonzero_for_annotation['pt_root_id'].isin(annotation_root_ids)]
nuclei_root0_for_annotation = nuclei_df[nuclei_df['pt_root_id'] == 0]

population_summary = pd.DataFrame([
    {
        'group': 'annotated nonzero roots',
        'row_count': int(len(nuclei_annotated)),
        'unique_root_count': int(nuclei_annotated['pt_root_id'].nunique(dropna=True)),
    },
    {
        'group': 'unlabeled nonzero roots',
        'row_count': int(len(nuclei_unlabeled_nonzero)),
        'unique_root_count': int(nuclei_unlabeled_nonzero['pt_root_id'].nunique(dropna=True)),
    },
    {
        'group': 'pt_root_id == 0',
        'row_count': int(len(nuclei_root0_for_annotation)),
        'unique_root_count': int(nuclei_root0_for_annotation['pt_root_id'].nunique(dropna=True)),
    },
])

display(population_summary)


,group,row_count,unique_root_count
0,annotated nonzero roots,7729,2061
1,unlabeled nonzero roots,16525,11812
2,pt_root_id == 0,11245,1


In [27]:
# Volume comparison for annotated, unlabeled nonzero, and root-0 groups
def annotation_volume_stats(df, label):
    volume = pd.to_numeric(df['volume'], errors='coerce').dropna()
    quantiles = volume.quantile([0.25, 0.50, 0.75, 0.90, 0.99]) if len(volume) else pd.Series(dtype='float64')
    return {
        'group': label,
        'count': int(volume.count()),
        '25th percentile': quantiles.get(0.25, None),
        'median': quantiles.get(0.50, None),
        '75th percentile': quantiles.get(0.75, None),
        '90th percentile': quantiles.get(0.90, None),
        '99th percentile': quantiles.get(0.99, None),
        'max': volume.max() if len(volume) else None,
    }

annotation_volume_summary = pd.DataFrame([
    annotation_volume_stats(nuclei_annotated, 'annotated nonzero roots'),
    annotation_volume_stats(nuclei_unlabeled_nonzero, 'unlabeled nonzero roots'),
    annotation_volume_stats(nuclei_root0_for_annotation, 'pt_root_id == 0'),
])

display(annotation_volume_summary)


,group,count,25th percentile,median,75th percentile,90th percentile,99th percentile,max
0,annotated nonzero roots,7729,0.097044,0.679311,64.590569,1227.587256,1437.206523,1677.171265
1,unlabeled nonzero roots,16525,0.145567,1.362355,52.657829,165.988608,347.592263,1258.916992
2,pt_root_id == 0,11245,0.093312,0.399375,2.892672,10.704753,373.985682,2389.182861


In [28]:
# Duplicate annotated roots
annotated_duplicate_root_ids = annotation_nucleus_counts[annotation_nucleus_counts > 1].sort_values(ascending=False).index.tolist()
annotation_identifying_columns = [
    column for column in annotation_df.columns
    if column == annotation_root_id_column
    or str(column).strip().lower() in {'type', 'subtypes', 'cell id', 'cell_id'}
    or str(column).strip().lower() in {'x', 'y', 'z'}
]
annotation_lookup = annotation_df.copy()
annotation_lookup['_annotation_root_id'] = pd.to_numeric(annotation_lookup[annotation_root_id_column], errors='coerce').astype('Int64')

annotated_duplicate_samples = []
for root_id in annotated_duplicate_root_ids[:15]:
    nucleus_rows = nuclei_df[nuclei_df['pt_root_id'] == root_id]
    annotation_rows = annotation_lookup[annotation_lookup['_annotation_root_id'] == root_id]
    annotation_record = annotation_rows[annotation_identifying_columns].head(1).to_dict('records')
    annotated_duplicate_samples.append({
        'pt_root_id': int(root_id),
        'annotation': annotation_record[0] if annotation_record else {},
        'nucleus_row_count': int(len(nucleus_rows)),
        'nucleus_volumes': pd.to_numeric(nucleus_rows['volume'], errors='coerce').dropna().tolist(),
        'pt_positions': nucleus_rows['pt_position'].tolist(),
    })

print(f"annotated roots with more than one nucleus row: {len(annotated_duplicate_root_ids)}")
print('compact sample of duplicate annotated roots:')
display(pd.DataFrame(annotated_duplicate_samples))


annotated roots with more than one nucleus row: 1644
compact sample of duplicate annotated roots:


,pt_root_id,annotation,nucleus_row_count,nucleus_volumes,pt_positions
0,648518346448199666,"{'Cell ID': 648518346448199666, 'x': 45808, 'y...",16,"[2.545551300048828, 0.0821145623922348, 0.0895...","[[45920, 48208, 1366], [45536, 47584, 1691], [..."
1,648518346462161795,"{'Cell ID': 648518346462161795, 'x': 44592, 'y...",16,"[1.0674892663955688, 0.0410572811961174, 0.067...","[[44720, 39952, 546], [44512, 39792, 553], [44..."
2,648518346443172484,"{'Cell ID': 648518346443172484, 'x': 47264, 'y...",16,"[0.343388170003891, 922.7735595703125, 0.03732...","[[47040, 39872, 879], [47264, 40160, 942], [47..."
3,648518346454880649,"{'Cell ID': 648518346454880649, 'x': 35728, 'y...",15,"[0.04478976130485535, 0.07838208228349686, 0.0...","[[35360, 58048, 1201], [35376, 58864, 1061], [..."
4,648518346443147652,"{'Cell ID': 648518346443147652, 'x': 42896, 'y...",14,"[0.06718464195728302, 3.083028554916382, 2.123...","[[42848, 42224, 564], [42880, 42144, 593], [42..."
5,648518346446645962,"{'Cell ID': 648518346446645962, 'x': 61280, 'y...",14,"[0.11570688337087631, 0.05971968173980713, 0.3...","[[61856, 67168, 1364], [62160, 66816, 1410], [..."
6,648518346429301812,"{'Cell ID': 648518346429301812, 'x': 39296, 'y...",14,"[0.06345216184854507, 0.11570688337087631, 4.3...","[[39344, 40064, 594], [39328, 40064, 591], [39..."
7,648518346438316205,"{'Cell ID': 648518346438316205, 'x': 39056, 'y...",14,"[0.09331200271844864, 0.7912857532501221, 715....","[[39936, 58448, 560], [39216, 62576, 875], [39..."
8,648518346442391506,"{'Cell ID': 648518346442391506, 'x': 33024, 'y...",13,"[0.07091712206602097, 0.7726233601570129, 0.07...","[[32272, 71008, 2050], [36656, 72704, 625], [3..."
9,648518346440948518,"{'Cell ID': 648518346440948518, 'x': 40928, 'y...",13,"[0.05225472152233124, 1383.8841552734375, 0.03...","[[40752, 54016, 1409], [40928, 54208, 1332], [..."


In [29]:
# Annotation comparison summary facts only
annotated_zero_count = int((annotation_nucleus_counts == 0).sum())
annotated_one_count = int((annotation_nucleus_counts == 1).sum())
annotated_multi_count = int((annotation_nucleus_counts > 1).sum())
unlabeled_nonzero_unique_count = int(nuclei_unlabeled_nonzero['pt_root_id'].nunique(dropna=True))
unlabeled_nonzero_row_count = int(len(nuclei_unlabeled_nonzero))

annotated_median = annotation_volume_summary.loc[
    annotation_volume_summary['group'] == 'annotated nonzero roots', 'median'
].iloc[0]
unlabeled_median = annotation_volume_summary.loc[
    annotation_volume_summary['group'] == 'unlabeled nonzero roots', 'median'
].iloc[0]
annotated_p99 = annotation_volume_summary.loc[
    annotation_volume_summary['group'] == 'annotated nonzero roots', '99th percentile'
].iloc[0]
unlabeled_p99 = annotation_volume_summary.loc[
    annotation_volume_summary['group'] == 'unlabeled nonzero roots', '99th percentile'
].iloc[0]

print('Annotation-to-nuclei comparison summary:')
print(f"- annotation-to-nuclei overlap: {len(matched_annotation_root_ids)} of {len(annotation_root_ids)} unique annotation roots matched ({match_percentage:.2f}%)")
print(f"- annotated roots by nucleus row count: 0={annotated_zero_count}, 1={annotated_one_count}, >1={annotated_multi_count}")
print(f"- nonzero unlabeled nuclei population: {unlabeled_nonzero_row_count} rows across {unlabeled_nonzero_unique_count} unique roots")
print(f"- volume medians annotated vs unlabeled nonzero: {annotated_median} vs {unlabeled_median}")
print(f"- volume 99th percentiles annotated vs unlabeled nonzero: {annotated_p99} vs {unlabeled_p99}")


Annotation-to-nuclei comparison summary:
- annotation-to-nuclei overlap: 2061 of 2764 unique annotation roots matched (74.57%)
- annotated roots by nucleus row count: 0=703, 1=417, >1=1644
- nonzero unlabeled nuclei population: 16525 rows across 11812 unique roots
- volume medians annotated vs unlabeled nonzero: 0.6793113350868225 vs 1.3623552322387695
- volume 99th percentiles annotated vs unlabeled nonzero: 1437.2065234375002 vs 347.5922631835938
